# Fundamentos de Regresión Logística
### Estadística Computacional para la Toma de Decisiones

**Magíster en Ciencia de Datos e Inteligencia Artificial**
Universidad Andrés Bello

*Maidana, J.P. (2026). Apunte — versión Jupyter Notebook.*

---

Este notebook es la versión **ejecutable** del apunte. Cada concepto teórico va acompañado de código que puedes correr y modificar: la curva sigmoide está construida con `matplotlib`, los *odds* y *odds ratios* se calculan sobre ejemplos reales, el modelo de aprobación de crédito se ajusta de verdad con `scikit-learn` **y** `statsmodels`, y verás la frontera de decisión, la matriz de confusión y la curva ROC dibujadas a partir de un modelo entrenado.

> 💡 Busca los comentarios **`👉 EXPERIMENTA`**: marcan parámetros que vale la pena cambiar para ver cómo reacciona el modelo.

**Requisitos:** sólo `numpy`, `pandas`, `matplotlib`, `scikit-learn` y `statsmodels` — todo viene preinstalado en Google Colab, así que corre sin instalar nada.

## Configuración del entorno

Importamos librerías, fijamos una semilla para reproducibilidad y definimos un estilo de gráficos consistente. **Ejecuta esta celda primero.**

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

# Reproducibilidad
np.random.seed(42)

# Estilo de graficos consistente para todo el notebook
plt.rcParams["figure.figsize"] = (9, 5.5)
plt.rcParams["axes.grid"] = True
plt.rcParams["grid.alpha"] = 0.3
plt.rcParams["font.size"] = 11
plt.rcParams["axes.titlesize"] = 13
plt.rcParams["axes.titleweight"] = "bold"

print("Entorno listo. Versiones:")
print("  numpy      :", np.__version__)
print("  pandas     :", pd.__version__)
import sklearn, statsmodels
print("  sklearn    :", sklearn.__version__)
print("  statsmodels:", statsmodels.__version__)

# 1. Introducción

Hasta ahora hemos trabajado con regresión lineal, donde predecimos variables continuas como precios, ventas, o temperaturas. Pero muchos problemas importantes en ciencia de datos requieren predecir **categorías**: ¿el cliente comprará o no? ¿el paciente tiene la enfermedad? ¿el email es spam? Estos son problemas de **clasificación binaria**, y la regresión logística es la herramienta estadística fundamental para abordarlos.

La regresión logística puede parecer inicialmente confusa. A pesar de su nombre, no es realmente una regresión en el sentido tradicional sino un método de clasificación. Usa una transformación matemática ingeniosa, la **función sigmoide**, para convertir cualquier valor real en una probabilidad entre 0 y 1. Esta probabilidad luego se interpreta como la probabilidad de pertenecer a una clase.

Lo hermoso de la regresión logística es que mantiene gran parte de la simplicidad e interpretabilidad de la regresión lineal, pero adaptada a variables categóricas. Los coeficientes tienen interpretaciones claras en términos de **odds ratios**, una medida que cuantifica cuánto cambian las probabilidades cuando cambian las variables predictoras. Además, el modelo tiene supuestos menos restrictivos que la regresión lineal, haciéndolo robusto y ampliamente aplicable.

En este notebook construiremos intuición desde la regresión lineal hacia la logística. Entenderemos por qué necesitamos la función sigmoide y cómo interpretarla. Aprenderemos qué son los odds y los odds ratios. Revisaremos los supuestos (que son diferentes y menos estrictos que en regresión lineal), y finalmente discutiremos cuándo usar regresión logística versus otras alternativas.

> ### 🎯 Lo que dominarás al finalizar
> Entenderás por qué la regresión lineal falla para clasificación y cómo la logística lo soluciona. Sabrás interpretar coeficientes en términos de log-odds y odds ratios. Podrás ajustar modelos en Python y comunicar resultados efectivamente. Reconocerás cuándo la regresión logística es apropiada y cuándo considerar alternativas.

# 2. De Regresión Lineal a Logística

Para entender por qué necesitamos regresión logística, comencemos viendo qué sucede si intentamos usar regresión lineal para clasificación.

## 2.1 El Problema con Regresión Lineal

Supongamos que queremos predecir si un cliente comprará un producto (1 = compra, 0 = no compra) basándonos en su ingreso. Si aplicamos regresión lineal directamente, obtenemos una ecuación como:

$$y = \beta_0 + \beta_1 \cdot \text{ingreso}$$

El problema fundamental es que esta ecuación puede producir valores **menores que 0 o mayores que 1**. Por ejemplo, para ingresos muy bajos podríamos obtener $\hat{y} = -0.3$, y para ingresos muy altos $\hat{y} = 1.4$. ¿Qué significa una probabilidad de $-0.3$ o de $1.4$? No tiene sentido.

La relación entre la variable independiente y la probabilidad rara vez es lineal en todo el rango. Usualmente, cuando el ingreso es muy bajo, aumentarlo un poco incrementa bastante la probabilidad de compra. Pero cuando el ingreso ya es alto, aumentarlo más tiene poco efecto adicional. Esta es una relación de **forma S**, no lineal.

Veámoslo con datos reales antes de seguir con la teoría.

In [ ]:
from sklearn.linear_model import LinearRegression

# Datos sinteticos: ¿compra el cliente? (1=si, 0=no) segun su ingreso
np.random.seed(7)
n = 100
ingreso_demo = np.random.uniform(10, 90, n)                   # ingreso en miles USD
prob_real = 1 / (1 + np.exp(-(0.12 * ingreso_demo - 6)))      # proceso generador real
compra = (np.random.random(n) < prob_real).astype(int)

# Ajustamos REGRESION LINEAL a una variable binaria (justo lo que NO se debe hacer)
lin = LinearRegression().fit(ingreso_demo.reshape(-1, 1), compra)
grid = np.linspace(0, 110, 200).reshape(-1, 1)
pred_lineal = lin.predict(grid)

# ¿Cuantas predicciones se salen del rango valido [0, 1]?
fuera = ((pred_lineal < 0) | (pred_lineal > 1)).sum()
print(f"Predicciones fuera de [0,1]: {fuera} de {len(grid)}")
print(f"Prediccion minima: {pred_lineal.min():.2f}  (¡probabilidad negativa!)")
print(f"Prediccion maxima: {pred_lineal.max():.2f}  (¡probabilidad mayor que 1!)")

plt.scatter(ingreso_demo, compra, alpha=0.5, label="Datos observados (0/1)")
plt.plot(grid, pred_lineal, "r-", lw=2, label="Regresion lineal")
plt.axhline(0, color="gray", ls="--", lw=1)
plt.axhline(1, color="gray", ls="--", lw=1)
plt.axhspan(1, pred_lineal.max(), color="red", alpha=0.08)
plt.axhspan(pred_lineal.min(), 0, color="red", alpha=0.08)
plt.xlabel("Ingreso (miles USD)")
plt.ylabel("¿Compra? / Prediccion")
plt.title("El problema: la recta predice probabilidades imposibles")
plt.legend()
plt.show()

Las zonas rojas son predicciones imposibles. Además, la regresión lineal asume que los errores son normales con varianza constante; para una variable binaria 0/1 los errores solo pueden tomar dos valores posibles, violando completamente ese supuesto.

## 2.2 La Necesidad de una Transformación

Lo que necesitamos es una función que:

- **Tome cualquier valor real**: el lado derecho de nuestra ecuación ($\beta_0 + \beta_1 x_1 + \cdots$) puede producir cualquier número de $-\infty$ a $+\infty$.
- **Produzca valores entre 0 y 1**: ya que interpretamos la salida como una probabilidad.
- **Tenga forma de S**: capturando que los efectos son mayores en rangos medios y se aplanan en los extremos.
- **Sea matemáticamente conveniente**: permitiendo estimación e interpretación relativamente simples.

La **función logística (o sigmoide)** cumple perfectamente estos requisitos. Comparémosla con la recta:

In [ ]:
def sigmoid(z):
    # Funcion sigmoide (o logistica): mapea cualquier real al intervalo (0, 1)
    return 1 / (1 + np.exp(-z))

z = np.linspace(-6, 6, 300)

fig, ax = plt.subplots(figsize=(9, 5.5))
ax.plot(z, 0.5 + 0.12 * z, "r--", lw=2, label="Regresion lineal (se sale de [0,1])")
ax.plot(z, sigmoid(z), "b-", lw=2.5, label="Funcion sigmoide (siempre en [0,1])")

ax.axhline(0, color="gray", ls=":", lw=1)
ax.axhline(1, color="gray", ls=":", lw=1)
ax.axhspan(1, 1.6, color="red", alpha=0.06)
ax.axhspan(-0.6, 0, color="red", alpha=0.06)
ax.set_ylim(-0.6, 1.6)
ax.set_xlabel("z (combinacion lineal de predictores)")
ax.set_ylabel("Probabilidad P(y=1)")
ax.set_title("Por que la sigmoide: confina la salida al rango valido [0, 1]")
ax.annotate("Rango valido [0, 1]", xy=(-4.5, 0.5), fontsize=11, color="green", ha="center")
ax.legend(loc="center right")
plt.show()

# 3. La Función Sigmoide

La función sigmoide (también llamada logística) es el corazón de la regresión logística.

## 3.1 Definición Matemática

La función sigmoide se define como:

$$\sigma(z) = \frac{1}{1 + e^{-z}} = \frac{e^z}{1 + e^z}$$

donde $z$ es cualquier número real. En regresión logística, $z$ es nuestra combinación lineal de predictores:

$$z = \beta_0 + \beta_1 x_1 + \beta_2 x_2 + \cdots + \beta_p x_p$$

Por lo tanto, el modelo completo de regresión logística es:

$$P(y=1 \mid \mathbf{x}) = \frac{1}{1 + e^{-(\beta_0 + \beta_1 x_1 + \cdots + \beta_p x_p)}}$$

Esta ecuación nos da la probabilidad de que $y=1$ (la clase positiva) dados los valores de las variables independientes.

## 3.2 Propiedades de la Sigmoide

- **Rango**: para cualquier $z$, la sigmoide produce un valor entre 0 y 1. Cuando $z \to -\infty$, $\sigma(z) \to 0$; cuando $z \to +\infty$, $\sigma(z) \to 1$.
- **Punto medio**: cuando $z=0$, vale exactamente 0.5 (el punto de inflexión).
- **Simetría**: $\sigma(-z) = 1 - \sigma(z)$.
- **Derivada conveniente**: $\sigma'(z) = \sigma(z)\,(1-\sigma(z))$. Esto facilita enormemente la optimización.
- **Monotonía**: siempre creciente. A mayor $z$, mayor probabilidad predicha.

Verifiquemos numéricamente estos valores y la identidad de la derivada:

In [ ]:
# Tabla de valores: la mayor parte de la "accion" ocurre entre z=-2 y z=2
valores_z = [-5, -2, -1, 0, 1, 2, 5]
tabla = pd.DataFrame({
    "z": valores_z,
    "sigma(z)": [round(sigmoid(z_i), 3) for z_i in valores_z],
})
print(tabla.to_string(index=False))

# Verificamos la propiedad de la derivada: sigma'(z) = sigma(z)*(1-sigma(z))
z0 = 1.3
h = 1e-6
deriv_numerica = (sigmoid(z0 + h) - sigmoid(z0 - h)) / (2 * h)
deriv_formula = sigmoid(z0) * (1 - sigmoid(z0))
print(f"\nDerivada en z={z0}:  numerica={deriv_numerica:.6f}  formula={deriv_formula:.6f}")

# Verificamos la simetria: sigma(-z) = 1 - sigma(z)
print(f"Simetria: sigma(-2)={sigmoid(-2):.4f}  y  1-sigma(2)={1-sigmoid(2):.4f}")

Observa que la mayor parte de la acción ocurre entre $z=-2$ y $z=2$. Fuera de ese rango, las probabilidades están muy cerca de 0 o 1.

## 3.3 Interpretación Geométrica

La curva sigmoide tiene forma de S estirada. En los extremos (valores muy negativos o muy positivos de $z$) la curva es casi plana: cambios en $z$ tienen poco efecto en la probabilidad. En el centro (cerca de $z=0$) la curva es más pronunciada: pequeños cambios en $z$ producen cambios significativos en la probabilidad.

Esta forma refleja muchos fenómenos reales. Al decidir comprar según el ingreso: cuando el ingreso es muy bajo, aumentarlo un poco puede pasar al cliente de "definitivamente no puedo" a "quizás puedo"; cuando ya es alto, aumentarlo más no cambia mucho la decisión.

In [ ]:
# Grafico de referencia de la sigmoide con el punto de decision marcado
z = np.linspace(-6, 6, 200)
prob = sigmoid(z)

plt.figure(figsize=(10, 6))
plt.plot(z, prob, "b-", linewidth=2)
plt.axhline(y=0.5, color="r", linestyle="--", alpha=0.5)
plt.axvline(x=0, color="r", linestyle="--", alpha=0.5)
plt.xlabel("z (combinacion lineal)")
plt.ylabel("Probabilidad P(y=1)")
plt.title("Funcion Sigmoide")
plt.ylim(-0.05, 1.05)
plt.scatter([0], [0.5], color="red", s=100, zorder=5)
plt.annotate("Punto de decision\n(z=0, p=0.5)", xy=(0, 0.5), xytext=(0.6, 0.42), fontsize=10)
plt.show()

# 4. Odds y Odds Ratio

Para interpretar los coeficientes de regresión logística, necesitamos entender dos conceptos fundamentales: **odds** y **odds ratio**.

## 4.1 ¿Qué Son los Odds?

Los **odds** (momios o razón de probabilidades) son una forma alternativa de expresar probabilidades. Si la probabilidad de un evento es $p$, los odds de ese evento son:

$$\text{odds} = \frac{p}{1-p}$$

Los odds representan la razón entre la probabilidad de que ocurra el evento versus la probabilidad de que no ocurra:

- Si $p=0.5$, los odds son $\tfrac{0.5}{0.5} = 1$ ("odds de 1 a 1").
- Si $p=0.8$, los odds son $\tfrac{0.8}{0.2} = 4$ (4 veces más probable que ocurra a que no).
- Si $p=0.25$, los odds son $\tfrac{0.25}{0.75} = 0.33$ (menos probable que ocurra).

Los odds van de 0 a $\infty$. Odds mayores que 1 indican que el evento es más probable que no; menores que 1, lo contrario.

In [ ]:
# Odds = p/(1-p). Convierten probabilidades en "razon a favor vs en contra".
probs = [0.25, 0.5, 0.8, 0.9, 0.99]
demo_odds = pd.DataFrame({
    "p (probabilidad)": probs,
    "odds = p/(1-p)": [round(p / (1 - p), 3) for p in probs],
    "log-odds (logit)": [round(np.log(p / (1 - p)), 3) for p in probs],
})
print(demo_odds.to_string(index=False))

# Las dos transformaciones clave, visualizadas
p = np.linspace(0.001, 0.999, 300)
odds = p / (1 - p)
logit = np.log(odds)

fig, axes = plt.subplots(1, 2, figsize=(13, 5))
axes[0].plot(p, odds, color="purple", lw=2)
axes[0].axhline(1, color="gray", ls="--", lw=1)
axes[0].set_xlabel("Probabilidad p")
axes[0].set_ylabel("odds = p/(1-p)")
axes[0].set_title("Odds: van de 0 a infinito")
axes[0].set_ylim(0, 20)

axes[1].plot(p, logit, color="darkgreen", lw=2)
axes[1].axhline(0, color="gray", ls="--", lw=1)
axes[1].set_xlabel("Probabilidad p")
axes[1].set_ylabel("logit(p) = log(odds)")
axes[1].set_title("Log-odds: de -inf a +inf\n(esto es lo que el modelo predice linealmente)")
plt.tight_layout()
plt.show()

## 4.2 Conexión con la Regresión Logística

Aquí viene la magia matemática. Si tomamos el logaritmo natural de los odds, obtenemos el **logit**:

$$\text{logit}(p) = \log\!\left(\frac{p}{1-p}\right)$$

Resulta que el logit tiene una relación **lineal** con nuestros predictores. Específicamente:

$$\log\!\left(\frac{P(y=1\mid\mathbf{x})}{1-P(y=1\mid\mathbf{x})}\right) = \beta_0 + \beta_1 x_1 + \cdots + \beta_p x_p$$

Mientras las probabilidades tienen una relación no lineal (sigmoide) con los predictores, los log-odds tienen una relación lineal. Por eso el panel derecho de arriba es una recta.

## 4.3 Odds Ratio: Interpretando Coeficientes

El **odds ratio** es la razón entre dos odds. En regresión logística nos dice cómo cambian los odds cuando una variable independiente aumenta en una unidad, manteniendo las demás constantes. El odds ratio asociado al coeficiente $\beta_j$ es simplemente:

$$\text{OR}_j = e^{\beta_j}$$

- Si $\text{OR}_j = 1$: la variable no tiene efecto (los odds no cambian).
- Si $\text{OR}_j > 1$: la variable aumenta los odds. $\text{OR}=2$ significa que los odds se duplican.
- Si $\text{OR}_j < 1$: la variable disminuye los odds. $\text{OR}=0.5$ significa que se reducen a la mitad.

### 📌 Ejemplo: Aprobación de Crédito

Supongamos un modelo que predice aprobación de crédito según ingreso (en miles de USD) y antigüedad laboral (en años):

$$\log(\text{odds de aprobación}) = -3 + 0.05 \cdot \text{ingreso} + 0.2 \cdot \text{antigüedad}$$

Calculemos los odds ratios y la probabilidad para una persona concreta. **Este es el mismo modelo que ajustaremos desde datos en la Sección 5.**

In [ ]:
# Modelo "verdadero" del ejemplo:
#   log(odds de aprobacion) = -3 + 0.05*ingreso + 0.2*antiguedad
beta_0, beta_ingreso, beta_antig = -3, 0.05, 0.2

# Odds ratios = exp(coeficiente)
OR_ingreso = np.exp(beta_ingreso)
OR_antig = np.exp(beta_antig)
print("Odds ratios (interpretacion de cada coeficiente):")
print(f"  OR ingreso    = exp({beta_ingreso}) = {OR_ingreso:.3f}  "
      f"->  +{(OR_ingreso-1)*100:.1f}% en los odds por cada mil USD adicionales")
print(f"  OR antiguedad = exp({beta_antig})  = {OR_antig:.3f}  "
      f"->  +{(OR_antig-1)*100:.1f}% en los odds por cada anio adicional")

# Persona con ingreso = 50 (mil USD) y 5 anios de antiguedad
log_odds = beta_0 + beta_ingreso * 50 + beta_antig * 5
odds = np.exp(log_odds)
prob = odds / (1 + odds)
print(f"\nPersona con ingreso=50k y antiguedad=5 anios:")
print(f"  log-odds = {beta_0} + {beta_ingreso}*50 + {beta_antig}*5 = {log_odds:.2f}")
print(f"  odds     = exp({log_odds:.2f}) = {odds:.2f}")
print(f"  P(aprobacion) = {prob:.2f}  ->  {prob:.0%}")

> ### ⚠️ Cuidado con la Interpretación
> Los odds ratios son **multiplicativos, no aditivos**. Si $\text{OR}=2$, los odds se duplican, pero la probabilidad **NO** necesariamente se duplica. El efecto sobre la probabilidad depende del punto de partida. Por esto los odds ratios son más estables y fáciles de interpretar que los cambios en probabilidad.

Comprobémoslo: duplicar los odds (OR = 2) desde distintos puntos de partida da incrementos de probabilidad muy distintos.

In [ ]:
# "Duplicar los odds" NO es "duplicar la probabilidad".
# El efecto sobre la probabilidad depende del punto de partida.
def duplicar_odds(p):
    odds_nuevos = 2 * (p / (1 - p))      # aplicamos OR = 2
    return odds_nuevos / (1 + odds_nuevos)

print("Efecto de duplicar los odds (OR=2) segun la probabilidad inicial:")
for p0 in [0.1, 0.5, 0.8]:
    p1 = duplicar_odds(p0)
    print(f"  p inicial = {p0:.2f}  ->  odds x2  ->  p nueva = {p1:.3f}   "
          f"(NO {2*p0:.2f}, que seria duplicar la probabilidad)")

# 👉 EXPERIMENTA: cambia el factor 2 por 0.5 (OR<1) y observa como bajan las probabilidades.

# 5. El Modelo de Regresión Logística

Ahora que entendemos los conceptos fundamentales, veamos el modelo completo y cómo se estima.

## 5.1 Formulación del Modelo

El modelo se puede expresar de **tres formas equivalentes**:

**Forma 1 — Probabilidad:**
$$P(y=1\mid\mathbf{x}) = \frac{1}{1 + e^{-(\beta_0 + \beta_1 x_1 + \cdots + \beta_p x_p)}}$$

**Forma 2 — Log-odds (logit):**
$$\log\!\left(\frac{P(y=1\mid\mathbf{x})}{1-P(y=1\mid\mathbf{x})}\right) = \beta_0 + \beta_1 x_1 + \cdots + \beta_p x_p$$

**Forma 3 — Odds:**
$$\frac{P(y=1\mid\mathbf{x})}{1-P(y=1\mid\mathbf{x})} = e^{\beta_0 + \beta_1 x_1 + \cdots + \beta_p x_p}$$

Las tres dicen lo mismo. La forma 1 es más intuitiva, la forma 2 muestra la relación lineal con los predictores, y la forma 3 es útil para interpretar coeficientes como odds ratios.

## 5.2 Estimación: Máxima Verosimilitud

A diferencia de la regresión lineal (mínimos cuadrados), la regresión logística se estima por **máxima verosimilitud**: encontrar los coeficientes $\beta$ que maximizan la probabilidad de observar los datos que realmente observamos. La función de verosimilitud es:

$$L(\boldsymbol{\beta}) = \prod_{i=1}^{n} p_i^{y_i}(1-p_i)^{1-y_i}$$

donde $p_i = P(y_i=1\mid\mathbf{x}_i)$. En la práctica maximizamos el **log-likelihood** (más estable numéricamente). No hay solución cerrada, por lo que se usan métodos iterativos (Newton-Raphson, gradiente descendente).

## 5.3 Implementación en Python

Generamos datos sintéticos con **el mismo modelo verdadero del ejemplo anterior** y comprobamos si lográmos recuperar los coeficientes $(-3,\ 0.05,\ 0.2)$ a partir de los datos.

In [ ]:
# ---- Datos sinteticos coherentes con el ejemplo de la Seccion 4 ----
np.random.seed(42)
n = 200
ingreso = np.random.uniform(20, 100, n)      # miles de USD
antiguedad = np.random.uniform(0, 10, n)     # anios

# El MISMO modelo verdadero: log(odds) = -3 + 0.05*ingreso + 0.2*antiguedad
z = -3 + 0.05 * ingreso + 0.2 * antiguedad
prob = 1 / (1 + np.exp(-z))

# Variable binaria observada
aprobado = (np.random.random(n) < prob).astype(int)

df = pd.DataFrame({
    "ingreso": ingreso,
    "antiguedad": antiguedad,
    "aprobado": aprobado,
})
print(f"Observaciones: {len(df)}   |   Tasa de aprobacion: {df['aprobado'].mean():.1%}")
df.head()

In [ ]:
from sklearn.linear_model import LogisticRegression

# ===== scikit-learn: ajuste rapido =====
X = df[["ingreso", "antiguedad"]]
y = df["aprobado"]

modelo_sk = LogisticRegression()
modelo_sk.fit(X, y)

print("--- scikit-learn ---")
print(f"Intercepto: {modelo_sk.intercept_[0]:7.4f}   (verdadero: -3.00)")
verdaderos = {"ingreso": 0.05, "antiguedad": 0.20}
for nombre, c in zip(X.columns, modelo_sk.coef_[0]):
    print(f"  {nombre:11s}: {c:7.4f}   (verdadero: {verdaderos[nombre]:.2f})")
print("\nLos coeficientes estimados se acercan a los verdaderos: el ajuste funciona.")

In [ ]:
import statsmodels.api as sm

# ===== statsmodels: tabla estadistica completa + odds ratios =====
X_const = sm.add_constant(X)            # agrega la columna del intercepto
modelo_sm = sm.Logit(y, X_const).fit()  # ajuste por maxima verosimilitud
print(modelo_sm.summary())

print("\n--- Odds Ratios (exponencial de cada coeficiente) ---")
odds_ratios = np.exp(modelo_sm.params)
tabla_or = pd.DataFrame({
    "coeficiente": modelo_sm.params.round(4),
    "odds_ratio": odds_ratios.round(4),
})
print(tabla_or.to_string())

El `summary()` de `statsmodels` entrega valores $p$, intervalos de confianza y el pseudo-$R^2$, todo lo que necesitas para inferencia. Ahora visualicemos qué aprendió el modelo: la **frontera de decisión** en el espacio (ingreso, antigüedad).

In [ ]:
# Frontera de decision: donde el modelo predice p=0.5 (es decir z=0)
xx = np.linspace(20, 100, 200)
yy = np.linspace(0, 10, 200)
XX, YY = np.meshgrid(xx, yy)
# Usamos un DataFrame con los mismos nombres de columna que en el ajuste
malla = pd.DataFrame({"ingreso": XX.ravel(), "antiguedad": YY.ravel()})
ZZ = modelo_sk.predict_proba(malla)[:, 1].reshape(XX.shape)

plt.figure(figsize=(10, 6))
cf = plt.contourf(XX, YY, ZZ, levels=20, cmap="RdYlGn", alpha=0.65)
plt.colorbar(cf, label="P(aprobacion)")
plt.contour(XX, YY, ZZ, levels=[0.5], colors="black", linewidths=2)

aprob = df["aprobado"] == 1
plt.scatter(df.loc[aprob, "ingreso"], df.loc[aprob, "antiguedad"],
            c="darkgreen", edgecolors="k", s=30, label="Aprobado (1)")
plt.scatter(df.loc[~aprob, "ingreso"], df.loc[~aprob, "antiguedad"],
            c="darkred", edgecolors="k", s=30, label="Rechazado (0)")
plt.xlabel("Ingreso (miles USD)")
plt.ylabel("Antiguedad (anios)")
plt.title("Frontera de decision (linea negra: p = 0.5)")
plt.legend(loc="lower left")
plt.show()

# 6. Supuestos de Regresión Logística

La regresión logística tiene supuestos, pero son **menos restrictivos** que los de regresión lineal.

## 6.1 Supuestos Principales

1. **Variable dependiente binaria** (0 o 1). Existen extensiones para múltiples categorías.
2. **Independencia de observaciones**: deben ser independientes entre sí (igual que en regresión lineal).
3. **Linealidad en el logit**: el log-odds debe tener relación lineal con las variables continuas. Esto **no** significa que la probabilidad sea lineal, sino que el logit lo es.
4. **Ausencia de multicolinealidad severa**: variables predictoras no deben estar perfectamente correlacionadas.
5. **Muestra suficientemente grande**: como regla práctica, al menos 10–15 observaciones por variable y preferiblemente $\geq 100$ en total.

## 6.2 Lo Que NO Se Asume

A diferencia de la regresión lineal, la logística **NO** requiere:

- **Normalidad de residuos**: los errores no son normales y no se espera que lo sean.
- **Homocedasticidad**: la varianza de los errores cambia inherentemente con la probabilidad predicha, y eso es normal.
- **Linealidad en las variables originales**: solo requiere linealidad en el logit (puedes incluir polinomios o interacciones).

## 6.3 Verificación de Supuestos

Para multicolinealidad usamos **VIF** (igual que en regresión lineal). Para la linealidad en el logit, la prueba de **Box-Tidwell**: se añade el término $x \cdot \ln(x)$ por cada variable continua; si su coeficiente es significativo, la linealidad se viola.

In [ ]:
from statsmodels.stats.outliers_influence import variance_inflation_factor

# --- Multicolinealidad: VIF ---
# Se agrega una constante para que el VIF se calcule correctamente.
X_vif = sm.add_constant(df[["ingreso", "antiguedad"]])
vif = pd.DataFrame({
    "Variable": X_vif.columns,
    "VIF": [variance_inflation_factor(X_vif.values, i) for i in range(X_vif.shape[1])],
})
print("VIF (en los predictores, valores < 5 indican baja multicolinealidad;")
print("la fila 'const' se ignora, su valor alto es normal):")
print(vif.round(3).to_string(index=False))

In [ ]:
# --- Linealidad en el logit: prueba de Box-Tidwell ---
# Agregamos el termino x*ln(x) por cada variable continua.
# Si su p-valor es < 0.05, hay evidencia de que se viola la linealidad en el logit.
bt = df[["ingreso", "antiguedad"]].copy()
bt["ingreso_x_ln"] = bt["ingreso"] * np.log(bt["ingreso"])
# antiguedad puede ser 0; acotamos por abajo para evitar log(0)
bt["antig_x_ln"] = bt["antiguedad"] * np.log(bt["antiguedad"].clip(lower=0.01))

bt_const = sm.add_constant(bt)
modelo_bt = sm.Logit(y, bt_const).fit(disp=False)

print("Terminos de Box-Tidwell (interesa su p-valor):")
print(modelo_bt.pvalues[["ingreso_x_ln", "antig_x_ln"]].round(4).to_string())
print("\nAmbos p > 0.05  ->  no hay evidencia de violacion de linealidad en el logit,")
print("lo esperado, ya que generamos los datos con un modelo lineal en el logit.")

> ### ⚠️ Supuestos Más Relajados
> La regresión logística es más robusta que la lineal porque no asume normalidad ni homocedasticidad, lo que la hace aplicable en muchas situaciones donde la lineal fallaría. Sin embargo, sigue siendo un modelo paramétrico con supuestos. Si tienes relaciones muy complejas o no lineales en el logit, métodos como árboles de decisión o redes neuronales pueden funcionar mejor.

# 7. Cuándo Usar Regresión Logística

Saber cuándo usarla es tan importante como saber cómo usarla.

## 7.1 Casos de Uso Ideales

La regresión logística es la elección apropiada cuando:

- **Variable dependiente binaria**: aprobado/rechazado, compra/no compra, éxito/fracaso, enfermo/sano.
- **Interpretabilidad importante**: los odds ratios son fáciles de comunicar a stakeholders no técnicos.
- **Probabilidades calibradas**: necesitas probabilidades bien calibradas, no solo clasificaciones.
- **Relación aproximadamente lineal en el logit**: los efectos son log-lineales en los odds.
- **Inferencia estadística**: necesitas valores $p$, intervalos de confianza y teoría bien desarrollada.

Antes de comparar con alternativas, evaluemos formalmente el modelo que entrenamos. El apunte insiste en **no mirar solo accuracy**.

In [ ]:
from sklearn.metrics import (confusion_matrix, classification_report,
                             roc_curve, roc_auc_score, ConfusionMatrixDisplay)

# Probabilidades y clases predichas (umbral 0.5)
prob_pred = modelo_sk.predict_proba(X)[:, 1]
clase_pred = (prob_pred >= 0.5).astype(int)

print(classification_report(y, clase_pred, target_names=["Rechazado", "Aprobado"]))
print(f"ROC-AUC: {roc_auc_score(y, prob_pred):.3f}")

fig, axes = plt.subplots(1, 2, figsize=(13, 5))

# Matriz de confusion
cm = confusion_matrix(y, clase_pred)
ConfusionMatrixDisplay(cm, display_labels=["Rechazado", "Aprobado"]).plot(
    ax=axes[0], cmap="Blues", colorbar=False)
axes[0].set_title("Matriz de confusion")
axes[0].grid(False)

# Curva ROC
fpr, tpr, _ = roc_curve(y, prob_pred)
axes[1].plot(fpr, tpr, "b-", lw=2, label=f"AUC = {roc_auc_score(y, prob_pred):.3f}")
axes[1].plot([0, 1], [0, 1], "k--", lw=1, label="Azar")
axes[1].set_xlabel("Tasa de falsos positivos")
axes[1].set_ylabel("Tasa de verdaderos positivos")
axes[1].set_title("Curva ROC")
axes[1].legend(loc="lower right")
plt.tight_layout()
plt.show()

# 👉 EXPERIMENTA: cambia el umbral 0.5 por 0.3 o 0.7 y observa como cambian
#    precision y recall en la matriz de confusion.

## 7.2 Comparación con Alternativas

| Método | Cuándo Preferirlo | Desventajas |
|---|---|---|
| **Reg. Logística** | Interpretabilidad, inferencia, relaciones log-lineales | No captura relaciones complejas |
| **Árboles de Decisión** | Relaciones no lineales, interacciones automáticas | Menos interpretable, *overfitting* |
| **Random Forest** | Predicción máxima, muchas variables | Caja negra, lento |
| **SVM** | Fronteras complejas, alta dimensión | Difícil interpretación |
| **Redes Neuronales** | Relaciones muy complejas, *big data* | Caja negra, requiere muchos datos |

Comparemos concretamente la regresión logística con un árbol de decisión sobre **nuestros mismos datos**:

In [ ]:
from sklearn.tree import DecisionTreeClassifier
from sklearn.model_selection import cross_val_score

# Comparacion concreta: regresion logistica vs arbol de decision
modelos = {
    "Regresion logistica": LogisticRegression(),
    "Arbol de decision":   DecisionTreeClassifier(max_depth=4, random_state=42),
}

print("Accuracy media (validacion cruzada 5-fold):")
for nombre, mod in modelos.items():
    scores = cross_val_score(mod, X, y, cv=5, scoring="accuracy")
    print(f"  {nombre:22s}: {scores.mean():.3f}  +/- {scores.std():.3f}")

print("\nEn datos generados por un proceso log-lineal, la regresion logistica suele igualar")
print("o superar al arbol, y ademas entrega coeficientes interpretables (odds ratios).")

## 7.3 Extensiones de Regresión Logística

- **Regresión logística multinomial**: para más de dos categorías no ordenadas (producto A, B o C).
- **Regresión logística ordinal**: para categorías ordenadas (muy insatisfecho → muy satisfecho).
- **Regresión logística regularizada**: añade penalización L1 (Lasso) o L2 (Ridge) para prevenir *overfitting* con muchas variables.

> ### 📌 Casos de Uso Reales
> **Marketing**: predecir si un cliente responderá a una campaña según demografía e historial.
> **Medicina**: predecir si un paciente desarrollará una enfermedad según factores de riesgo.
> **Finanzas**: predecir si un solicitante entrará en *default* según ingreso y ratios de deuda.
> **Recursos Humanos**: predecir si un empleado abandonará la empresa (*churn*).
> **E-commerce**: predecir si un visitante realizará una compra según fuente de tráfico y comportamiento.

## 7.4 Cuándo NO Usar Regresión Logística

- **Relaciones altamente no lineales** en el logit → GAM o machine learning funcionan mejor.
- **Muchas interacciones complejas** → árboles o *ensemble methods* las capturan automáticamente.
- **Solo importa predicción máxima** (no interpretabilidad) → ML más sofisticado puede dar mejor *accuracy*.
- **Datos muy desbalanceados** ($<5\%$ de una clase) → considera técnicas de balanceo o métodos especializados.
- **Variables dependientes con estructura compleja** (multi-etiqueta) → métodos especializados.

# 8. Resumen y Mejores Prácticas

## 8.1 Conceptos Fundamentales

La regresión logística usa la función sigmoide para transformar una combinación lineal de predictores en probabilidades entre 0 y 1, creando la característica forma de S. Los coeficientes tienen interpretación directa en términos de log-odds; exponenciándolos obtenemos odds ratios. El modelo se estima por máxima verosimilitud (no mínimos cuadrados), lo que requiere optimización numérica iterativa pero proporciona estimaciones con buenas propiedades estadísticas.

## 8.2 Flujo de Trabajo Recomendado

1. **Exploración**: examina la distribución de la variable dependiente (¿desbalanceada?) y las relaciones con los predictores.
2. **Preparación**: trata faltantes, codifica categóricas, divide train/test, considera estandarizar si usarás regularización.
3. **Ajuste inicial**: comienza con todas las variables relevantes; usa `statsmodels` para estadísticas detalladas.
4. **Interpretación**: examina coeficientes, valores $p$ y odds ratios. ¿Tienen sentido los signos y magnitudes?
5. **Evaluación**: usa métricas apropiadas (accuracy, precision, recall, F1, ROC-AUC) según el problema de negocio.
6. **Diagnóstico**: verifica supuestos (multicolinealidad, linealidad en logit) e identifica observaciones influyentes.
7. **Refinamiento**: elimina variables no significativas, prueba transformaciones o interacciones.

## 8.3 Errores Comunes a Evitar

- **Confundir odds con probabilidades**: un OR de 2 NO significa que la probabilidad se duplica.
- **Interpretar sin considerar escala**: los coeficientes dependen de las unidades; estandariza para comparar importancia.
- **Ignorar desbalanceo de clases**: con una clase muy rara, el modelo tenderá a predecir siempre la mayoritaria.
- **Usar solo accuracy**: en problemas desbalanceados es engañosa (95% accuracy prediciendo siempre "No" si solo 5% son "Sí").
- **No verificar supuestos**: aunque menos restrictivos, existen.
- **Sobreinterpretar coeficientes pequeños**: un OR de 1.05 puede ser significativo pero prácticamente irrelevante.

> ### 💡 Cierre
> La regresión logística es una herramienta elegante que extiende la simplicidad de la regresión lineal a problemas de clasificación. Su fortaleza reside en el balance entre poder predictivo e interpretabilidad. Los odds ratios permiten comunicar resultados claramente a audiencias no técnicas, y sus supuestos menos restrictivos la hacen robusta en muchas situaciones. Aunque métodos más sofisticados pueden superarla en accuracy pura, la regresión logística sigue siendo la primera opción cuando la interpretabilidad importa. Dominarla es esencial para cualquier científico de datos y servirá como base para entender modelos más complejos.

---
## Referencias bibliográficas

- Agresti, A. (2019). *An introduction to categorical data analysis* (3rd ed.). Wiley.
- Hosmer, D. W., Lemeshow, S., & Sturdivant, R. X. (2013). *Applied logistic regression* (3rd ed.). Wiley.
- James, G., Witten, D., Hastie, T., & Tibshirani, R. (2021). *An introduction to statistical learning* (2nd ed.). Springer.
- King, G., & Zeng, L. (2001). Logistic regression in rare events data. *Political Analysis*, 9(2), 137–163.
- Pedregosa, F., et al. (2011). Scikit-learn: Machine learning in Python. *Journal of Machine Learning Research*, 12, 2825–2830.